# Notebook 04 — Módulo M5: Ranking Híbrido (RRF)

**Disciplina:** Inteligência Artificial — FACOM/UFMS — 2026/1
**Tema:** IA para Reconhecimento de Vocalizações de Psittacidae

**Módulo de aprofundamento M5:** Ranking Híbrido sparse+dense via
**Reciprocal Rank Fusion (RRF)** (Cormack, Clarke & Buettcher, 2009).

### Motivação
BM25 e TF-IDF-KNN têm pontos fortes complementares:
- BM25 → alta precisão para termos técnicos exatos ("psittacidae", "mfcc")
- TF-IDF KNN → melhor cobertura via bigrams e normalização global

RRF combina as duas listas **sem normalização de scores** (que seria problemática
por serem em escalas diferentes), usando apenas a **posição** de cada documento:

$$\text{RRF}(d) = \sum_{r \in \{bm25, tfidf\}} \frac{1}{k + \text{rank}_r(d)}$$

onde *k = 60* é a constante empírica de Cormack et al.


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, "..")

from src.utils import load_corpus, load_queries, write_trec_run
from src.retrievers import BM25Retriever, TFIDFKNNRetriever, HybridRRFRetriever


In [2]:
docs = load_corpus("../data/corpus.jsonl")
print(f"Corpus: {len(docs)} documentos")


Corpus: 1092 documentos


## 1. Construção dos três retrievers

In [3]:
print("Construindo BM25...")
bm25 = BM25Retriever(docs, k1=1.5, b=0.75)
print("Construindo TF-IDF KNN...")
tfidf = TFIDFKNNRetriever(docs)
print("Construindo Híbrido RRF (k=60)...")
hybrid = HybridRRFRetriever(docs, rrf_k=60)
print("Pronto.")


Construindo BM25...
Construindo TF-IDF KNN...


Construindo Híbrido RRF (k=60)...


Pronto.


## 2. Comparação top-10 para query de exemplo

In [4]:
q = "deep learning parrot vocalization recognition mel spectrogram"
print(f"Query: {q!r}\n")

bm25_res   = bm25.search(q, k=10)
tfidf_res  = tfidf.search(q, k=10)
hybrid_res = hybrid.search(q, k=10)

for label, results in [("BM25", bm25_res), ("TF-IDF KNN", tfidf_res), ("Híbrido RRF", hybrid_res)]:
    print(f"--- {label} ---")
    for rank, (doc_id, sc) in enumerate(results[:5], 1):
        doc = next(d for d in docs if d["arxiv_id"] == doc_id)
        print(f"  {rank}. [{sc:.4f}] {doc['title'][:70]}")
    print()


Query: 'deep learning parrot vocalization recognition mel spectrogram'

--- BM25 ---
  1. [13.3035] Detection of ground parrot vocalisation: A multiple instance learning 
  2. [12.5332] Classification of Sound using Convolutional Neural Networks
  3. [11.3997] A Novel Deep Learning Approach for Classification of Bird Sound Using 
  4. [10.4565] Automatic Classification of Bird Sounds: Using MFCC and Mel Spectrogra
  5. [10.1850] Improving Bird Vocalization Recognition in Open-Set Cross-Corpus Scena

--- TF-IDF KNN ---
  1. [0.1487] Beyond amplitude: Phase integration in bird vocalization recognition w
  2. [0.1322] Improving Bird Vocalization Recognition in Open-Set Cross-Corpus Scena
  3. [0.0981] Classification of Sound using Convolutional Neural Networks
  4. [0.0951] Implementation of Constant-Q Transform (CQT) and Mel Spectrogram to co
  5. [0.0923] Improved Bird Sound Classification Based on Deep Cascade Feature

--- Híbrido RRF ---
  1. [0.0320] Classification of Sound using Con

## 3. Geração dos três arquivos de run para avaliação

In [5]:
queries = load_queries("../eval/queries.tsv")
runs_dir = Path("runs")
runs_dir.mkdir(exist_ok=True)

for name, ret in [("bm25", bm25), ("knn_tfidf", tfidf), ("hybrid_rrf", hybrid)]:
    results_all = {row["qid"]: ret.search(row["text"], k=100)
                   for _, row in queries.iterrows()}
    write_trec_run(results_all, runs_dir / f"{name}.trec", system_name=name)
    print(f"Salvo: runs/{name}.trec")


Salvo: runs/bm25.trec


Salvo: runs/knn_tfidf.trec


Salvo: runs/hybrid_rrf.trec


## 4. Avaliação quantitativa

In [6]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "../eval/evaluate.py",
     "--qrels", "../eval/qrels.tsv",
     "--runs",
     "runs/bm25.trec", "runs/knn_tfidf.trec", "runs/hybrid_rrf.trec",
     "--k", "10"],
    capture_output=True, text=True, cwd=Path(".").resolve()
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[:500])



=== bm25.trec ===
qid   P@10   R@10     AP  nDCG@10
q01 0.0000 0.0000 0.0000   0.0000
q02 0.0000 0.0000 0.0000   0.0000
q03 0.0000 0.0000 0.0000   0.0000
q04 0.0000 0.0000 0.0000   0.0000
q05 0.0000 0.0000 0.0000   0.0000
q06 0.0000 0.0000 0.0000   0.0000
q07 0.0000 0.0000 0.0000   0.0000
q08 0.0000 0.0000 0.0000   0.0000
q09 0.0000 0.0000 0.0000   0.0000
q10 0.0000 0.0000 0.0000   0.0000
q11 0.0000 0.0000 0.0000   0.0000
q12 0.0000 0.0000 0.0000   0.0000
q13 0.0000 0.0000 0.0000   0.0000
q14 0.0000 0.0000 0.0000   0.0000
q15 0.0000 0.0000 0.0000   0.0000

Medias:
P@10      0.0000
R@10      0.0000
AP        0.0000
nDCG@10   0.0000

=== knn_tfidf.trec ===
qid   P@10   R@10     AP  nDCG@10
q01 0.0000 0.0000 0.0000   0.0000
q02 0.0000 0.0000 0.0000   0.0000
q03 0.0000 0.0000 0.0000   0.0000
q04 0.0000 0.0000 0.0000   0.0000
q05 0.0000 0.0000 0.0000   0.0000
q06 0.0000 0.0000 0.0000   0.0000
q07 0.0000 0.0000 0.0000   0.0000
q08 0.0000 0.0000 0.0000   0.0000
q09 0.0000 0.0000 0.0000   0.0

## 5. Análise dos resultados

**Por que o híbrido tende a superar os componentes individuais?**

O RRF é robusto porque:
1. Documentos que aparecem alto em *ambas* as listas recebem pontuação dobrada.
2. A divisão por *k + rank* penaliza posições baixas sem eliminar documentos presentes em apenas uma lista.
3. Não requer calibração de escala entre BM25 e TF-IDF.

**Análise qualitativa — query difícil:**
Identifique uma query em que o híbrido recupera documentos que nenhum dos componentes recuperava individualmente no top-10. Isso demonstra a complementaridade dos retrievers.


In [7]:
# Analisa a interseção e diferença entre os tops de cada modelo
def top_ids(results, k=10):
    return {doc_id for doc_id, _ in results[:k]}

for _, row in queries.iterrows():
    b = set(d for d,_ in bm25.search(row["text"], k=10))
    t = set(d for d,_ in tfidf.search(row["text"], k=10))
    h = set(d for d,_ in hybrid.search(row["text"], k=10))
    only_hybrid = h - b - t
    if only_hybrid:
        print(f"[{row['qid']}] {row['text']!r}")
        for doc_id in only_hybrid:
            doc = next(d for d in docs if d["arxiv_id"] == doc_id)
            print(f"  → só no híbrido: {doc['title'][:70]}")
        print()


[q01] 'deep learning for parrot call recognition'
  → só no híbrido: Machine Learning for Identifying an Endengered Brazilian Psittacidae S
  → só no híbrido: Deep learning techniques for bird chirp recognition task

[q03] 'mel spectrogram bird sound species identification'
  → só no híbrido: Wild Bird Species Identification Based on a Lightweight Model With Fre

[q04] 'convolutional neural network avian vocalization detection'
  → só no híbrido: Multi-Label Bird Species Classification Using Sequential Aggregation S



[q06] 'few-shot learning bird sound recognition'
  → só no híbrido: Pretraining Representations for Bioacoustic Few-shot Detection using S
  → só no híbrido: Few-Shot Bioacoustic Event Detection with Frame-Level Embedding Learni

[q07] 'passive acoustic monitoring parrot species'
  → só no híbrido: Contact calls of island Brown-throated Parakeets exhibit both characte



[q09] 'transformer model bird species identification audio'
  → só no híbrido: Deep Learning-based Environmental Sound Classification Using Feature F
  → só no híbrido: Effects of landscape and distance in automatic audio based bird specie

[q10] 'BirdNET deep learning bird sound classification'
  → só no híbrido: AudioProtoPNet: An interpretable deep learning model for bird sound cl
  → só no híbrido: <scp>BirdNET</scp>: applications, performance, pitfalls and future opp

[q13] 'ecoacoustics parrot species diversity soundscape'
  → só no híbrido: Accounting for both automated recording unit detection space and signa

